# Notebook 19: LLM Evaluation & Benchmarks Deep Dive

**Goal**: Master the landscape of LLM evaluation -- benchmarks, contamination, Goodhart's Law, and LLM-as-judge -- so you can discuss evaluation intelligently in frontier AI lab interviews.

**Prerequisites**: Familiarity with transformers, language models, basic ML evaluation metrics.

---

## 1. Self-Quiz (Answer Before Reading!)

Test yourself first. Write your answers in the cell below, then check against the notebook content.

1. **Name 5 LLM benchmarks.** What does each measure?
2. **What is benchmark contamination?** Why does it matter?
3. **What is Goodhart's Law for benchmarks?** Give a concrete example.
4. **What is LLM-as-Judge?** What are its biases?
5. **Why is Chatbot Arena considered the gold standard?** What makes it different from static benchmarks?
6. **What is the difference between GSM8K and MATH?** Why does saturation matter?
7. **What is SWE-bench?** Why is it considered harder than HumanEval?

**Your answers here** (double-click to edit):

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...
7. ...

---
## 2. Setup

In [ ]:
!pip install -q torch transformers datasets matplotlib pandas

In [ ]:
import torch
import json
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 3. The Eval Landscape: Comprehensive Benchmark Reference

*(All scores approximate, June 2026 — check live leaderboards before interviews.)*

### Reasoning & Knowledge Benchmarks

| Benchmark | What It Measures | Format | SOTA (June 2026, approx.) | Limitations |
|-----------|-----------------|--------|----------------|-------------|
| **MMLU** | Broad knowledge across 57 subjects (STEM, humanities, social science, etc.) | 4-option MCQ, 14K questions | ~90%+ (GPT-5.5, Claude Opus 4.8, Gemini 3.1 Pro) | **Largely saturated**. Noisy labels (~6% error rate). Contamination concerns. |
| **MMLU-Pro** | Harder version of MMLU with 10 answer choices (vs. 4), requiring more reasoning | 10-option MCQ | ~90% (frontier models) — effectively saturated | Harder to game via elimination, but still MCQ format. |
| **GPQA-Diamond** | Graduate-level (PhD) science questions (physics, chemistry, biology) | MCQ, expert-written | ~94% (Gemini 3.1 Pro: 94.3%) — effectively saturated; expert humans ~65% | Small dataset. Models now far exceed domain experts, so it no longer differentiates frontier models. |
| **ARC-AGI** | Abstraction and Reasoning Corpus -- visual pattern completion tasks | Grid-based pattern matching | v1 ~50%+ with heavy engineering; **ARC-AGI-2 is the active version** and remains unsaturated | Tests a very specific kind of reasoning. Not "AGI" despite the name. |

### Math Benchmarks

| Benchmark | What It Measures | Format | SOTA (June 2026, approx.) | Limitations |
|-----------|-----------------|--------|----------------|-------------|
| **GSM8K** | Grade-school math word problems | Free-form, 8.5K problems | ~97%+ (frontier models) | **Saturated.** No longer differentiates frontier models. |
| **MATH** | Competition math (AMC/AIME level) | Free-form, 12.5K problems | ~90%+ (frontier models) | Saturated for top models. |
| **AIME 2024/2025** | American Invitational Mathematics Examination -- hard competition math | 15 problems, integer answer 0-999 | **Saturated** — ~100% with tools; dropped from several aggregate indices | Very small test set (15 problems). High variance. |
| **FrontierMath** | Expert-level research math created by professional mathematicians | Free-form, peer-reviewed | ~52% on Tiers 1-3 (GPT-5.5 Pro) | Hardest tier still largely unsolved. Limited public access. |

### Code Benchmarks

| Benchmark | What It Measures | Format | SOTA (June 2026, approx.) | Limitations |
|-----------|-----------------|--------|----------------|-------------|
| **HumanEval** | Python function generation from docstrings | 164 problems, pass@1 | ~95%+ | **Saturated.** Contaminated. Too easy. |
| **HumanEval+** | Cleaned version with more rigorous test cases | 164 problems, stricter tests | ~90% | Better than original but same base problems. |
| **MBPP** | Mostly Basic Python Problems | ~1000 problems | ~90%+ | Simple problems, mostly saturated. |
| **SWE-bench Verified** | Human-verified subset of real GitHub issues (500 problems) | Full repo context, patch generation | Legacy scores ~95% — **retired by OpenAI in Feb 2026** (contamination + broken tests) | No longer the standard; treat scores as legacy numbers. |
| **SWE-bench Pro** | Successor benchmark of real GitHub issues — **the current coding standard** | Full repo context, patch generation | ~80% (frontier models) | Newer; scaffolding-dependent like its predecessor. |
| **Terminal-Bench 2.0** | Agentic terminal/command-line tasks | Agent operating in a sandboxed terminal | ~83% (frontier models) | Newer standard alongside SWE-bench Pro. |
| **LiveCodeBench** | Continuously updated competitive programming problems | Code generation + execution | ~70-80% (easy), ~30% (hard) | Avoids contamination via temporal cutoff. |
| **BigCodeBench** | Complex, multi-library coding tasks | Function-level + full program | ~60-70% | Tests practical coding beyond algorithms. |

### Chat & Preference Benchmarks

| Benchmark | What It Measures | Format | SOTA (June 2026, approx.) | Limitations |
|-----------|-----------------|--------|----------------|-------------|
| **Chatbot Arena (LMArena)** | Overall user preference via blind A/B comparisons | Bradley-Terry-based Arena Score from human votes (colloquially still called "Elo" — LMArena replaced online Elo with a Bradley-Terry MLE fit in Dec 2023) | Top ~1500 (Claude Opus 4.6/4.7 Thinking variants held top-3 at ~1498-1504, June 2026) | **Gold standard.** Slow to collect data. Biased toward English, certain user demographics. |
| **AlpacaEval 2.0** | Automated win rate vs. reference model (GPT-4 Turbo) | LC (length-controlled) win rate | ~70%+ (frontier) | Length bias even with LC correction. Single reference model. |
| **MT-Bench** | Multi-turn conversation quality across 8 categories | LLM judge scores 1-10 | ~9.5/10 (frontier) | Saturating. Small (80 questions). Judge model biases. |
| **Arena-Hard** | Harder subset of Arena questions, auto-evaluated | Win rate vs. baseline | ~85%+ (frontier) | Automated, so inherits LLM judge biases. |
| **WildBench** | Real user queries from the wild, challenging and diverse | LLM-judged pairwise comparison | ~75% win rate vs. GPT-4 | Newer, still being validated. |

### Instruction Following

| Benchmark | What It Measures | Format | SOTA (June 2026, approx.) | Limitations |
|-----------|-----------------|--------|----------------|-------------|
| **IFEval** | Whether model follows precise formatting constraints ("respond in exactly 3 sentences", "use bullet points") | Constraint verification | ~85-90% (frontier) | Tests surface-level compliance, not deep understanding. |

### Safety & Factuality

| Benchmark | What It Measures | Format | SOTA (June 2026, approx.) | Limitations |
|-----------|-----------------|--------|----------------|-------------|
| **TruthfulQA** | Resistance to common misconceptions and falsehoods | MCQ + generation | ~75-85% | Small, somewhat arbitrary question selection. |
| **BBQ** | Bias in question answering across 11 social categories | MCQ with ambiguous contexts | Varies by category | Narrow definition of bias. |
| **HarmBench** | Robustness to diverse attack types | Attack success rate | Varies by defense | Attack-dependent, constantly evolving. |
| **StrongREJECT** | Quality of refusals (not just whether model refuses, but how well) | Rubric-based scoring | Varies | Newer benchmark, less established. |
| **SimpleQA** | Factual accuracy on simple questions (hallucination test) | Correct/Incorrect/Not Attempted | ~40-50% (many models hallucinate heavily) | Only tests factual recall, not reasoning. |

### Grand Challenge

| Benchmark | What It Measures | Format | SOTA (June 2026, approx.) | Limitations |
|-----------|-----------------|--------|----------------|-------------|
| **Humanity's Last Exam (HLE)** | Expert-level questions across all academic disciplines, designed to be extremely hard for current AI | Open-ended, expert-created | ~53% no-tools / ~64% with tools (frontier models) | Designed to be extremely hard, but progress has been rapid. Small dataset. |

### Key Takeaways for Interviews

1. **Saturated benchmarks** (GSM8K, HumanEval, MMLU — and by mid-2026 also GPQA-Diamond, MMLU-Pro, and AIME) are no longer useful for comparing frontier models
2. **Chatbot Arena** is the closest thing to ground truth for overall model quality
3. **Contamination** is a constant concern — SWE-bench Verified was retired by OpenAI in Feb 2026 over contamination and broken tests; newer benchmarks (SWE-bench Pro, Terminal-Bench 2.0, LiveCodeBench, FrontierMath) address this via harder curation, temporal cutoffs, or restricted access
4. **No single benchmark tells the full story** -- always look at a suite of evaluations
5. **The frontier is moving fast** -- all numbers above are approximate (June 2026); re-check live leaderboards before interviews

**Insider Tip:** Chatbot Arena (LMArena) is the gold standard that frontier labs pay close attention to. Arena-style blind pairwise comparison with human raters is a widely used evaluation pattern across the industry. If you mention only one benchmark in an interview, mention Arena.

**Insider Tip:** Benchmark saturation is a real problem. MMLU is near-solved. The frontier has moved to MMLU-Pro, GPQA (PhD-level), and FrontierMath (research-level). Knowing this shows you're current.

**Insider Tip:** SimpleQA (OpenAI, 2024) is surprisingly useful -- it tests factual accuracy with simple questions. Models hallucinate 10-40% of the time even on easy factual questions. This is a great data point for safety discussions.

---
## 4. Build a Mini Eval Harness

Let's build a simple evaluation pipeline from scratch. This is the kind of thing you might be asked to implement in an interview.

### 4a. Load GSM8K and Implement Answer Extraction

In [ ]:
# Load a subset of GSM8K
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
print(f"Total GSM8K test examples: {len(gsm8k)}")

# Look at a sample
sample = gsm8k[0]
print(f"\nQuestion:\n{sample['question']}")
print(f"\nAnswer:\n{sample['answer']}")

In [ ]:
def extract_gsm8k_answer(answer_text: str) -> str:
    """
    Extract the final numerical answer from GSM8K format.
    GSM8K answers end with '#### <number>'
    """
    # GSM8K gold answers have '#### <answer>' format
    match = re.search(r'####\s*([\d,\.\-]+)', answer_text)
    if match:
        return match.group(1).replace(',', '').strip()
    return answer_text.strip()


def extract_model_answer(model_output: str) -> str:
    """
    Extract numerical answer from model's free-form response.
    Tries multiple patterns: 'The answer is X', 'X', boxed answers, etc.
    """
    # Pattern 1: "The answer is X" or "the answer is X"
    match = re.search(r'[Tt]he\s+answer\s+is\s*[:\s]*\$?([\d,\.\-]+)', model_output)
    if match:
        return match.group(1).replace(',', '').strip()
    
    # Pattern 2: "#### X" (GSM8K format)
    match = re.search(r'####\s*([\d,\.\-]+)', model_output)
    if match:
        return match.group(1).replace(',', '').strip()
    
    # Pattern 3: \boxed{X} (LaTeX format)
    match = re.search(r'\\boxed\{([\d,\.\-]+)\}', model_output)
    if match:
        return match.group(1).replace(',', '').strip()
    
    # Pattern 4: Last number in the text
    numbers = re.findall(r'[\d,]+\.?\d*', model_output)
    if numbers:
        return numbers[-1].replace(',', '').strip()
    
    return model_output.strip()


def exact_match(predicted: str, gold: str) -> bool:
    """Check if predicted answer exactly matches gold answer."""
    try:
        return float(predicted) == float(gold)
    except ValueError:
        return predicted.strip().lower() == gold.strip().lower()


# Test the answer extraction
test_cases = [
    ("Natalia sold 48/2 = 24 clips in May.\n#### 72", "72"),
    ("So the answer is 42.", "42"),
    ("Therefore, \\boxed{150}", "150"),
    ("Step 1: 5+3=8. Step 2: 8*2=16. The total is 16.", "16"),
]

for text, expected in test_cases:
    extracted = extract_model_answer(text)
    status = "PASS" if exact_match(extracted, expected) else "FAIL"
    print(f"[{status}] Input: '{text[:50]}...' -> Extracted: '{extracted}', Expected: '{expected}'")

### 4b. Run Evaluation with a Small Model

In [ ]:
# Use a small model for demonstration (in practice, you'd use a larger model)
# We'll use a small model and evaluate on a tiny subset to keep it fast
MODEL_NAME = "microsoft/phi-2"  # Small but capable model

print("Loading model... (this may take a minute)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
if not torch.cuda.is_available():
    model = model.to(device)
model.eval()
print("Model loaded!")

In [ ]:
def generate_answer(question: str, model, tokenizer, max_new_tokens=256) -> str:
    """Generate an answer from the model for a math question."""
    prompt = f"""Solve the following math problem step by step. End with 'The answer is <number>'.

Question: {question}

Solution:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated part
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated


def evaluate_gsm8k(dataset, model, tokenizer, num_samples=20):
    """
    Evaluate model on GSM8K subset.
    Returns accuracy and detailed results.
    """
    results = []
    correct = 0
    
    # Sample a subset
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    
    for i, idx in enumerate(indices):
        example = dataset[idx]
        question = example['question']
        gold_answer = extract_gsm8k_answer(example['answer'])
        
        # Generate model answer
        model_output = generate_answer(question, model, tokenizer)
        predicted_answer = extract_model_answer(model_output)
        
        is_correct = exact_match(predicted_answer, gold_answer)
        if is_correct:
            correct += 1
        
        results.append({
            'question': question[:80] + '...',
            'gold': gold_answer,
            'predicted': predicted_answer,
            'correct': is_correct
        })
        
        if (i + 1) % 5 == 0:
            print(f"  Evaluated {i+1}/{num_samples} (running accuracy: {correct/(i+1):.1%})")
    
    accuracy = correct / len(results)
    return accuracy, results


# Run evaluation
num_samples = 20
print("Evaluating on GSM8K subset...")
accuracy, results = evaluate_gsm8k(gsm8k, model, tokenizer, num_samples=num_samples)

num_correct = sum(r['correct'] for r in results)
print(f"\n{'='*60}")
print(f"GSM8K Accuracy: {accuracy:.1%} ({num_correct}/{len(results)})")
print(f"{'='*60}")

# Show some results
results_df = pd.DataFrame(results)
print("\nSample results:")
print(results_df[['gold', 'predicted', 'correct']].head(10).to_string())

### 4c. Mini LLM-as-Judge

In [ ]:
def llm_judge(question: str, response_a: str, response_b: str, 
              judge_model, judge_tokenizer, max_new_tokens=200) -> dict:
    """
    Use an LLM to judge which response is better.
    Returns a dict with the verdict and explanation.
    
    This implements the core idea behind MT-Bench, Chatbot Arena, 
    and AlpacaEval -- using a strong model to evaluate weaker models.
    """
    judge_prompt = f"""You are a fair judge evaluating two AI assistant responses.

Question: {question}

Response A:
{response_a}

Response B:
{response_b}

Which response is better? Consider accuracy, helpfulness, and clarity.
Output ONLY one of: "A is better", "B is better", or "Tie".
Then briefly explain why.

Verdict:"""
    
    inputs = judge_tokenizer(judge_prompt, return_tensors="pt", truncation=True, max_length=1024).to(judge_model.device)
    
    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=judge_tokenizer.eos_token_id
        )
    
    judgment = judge_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Parse the verdict by looking for an EXPLICIT verdict token, and treat
    # ambiguous output (both or neither found) as a tie.
    # NOTE: sloppy verdict parsing is itself a bias source in judge evals --
    # e.g., checking `"Response A" in judgment` before checking B silently
    # awards A every time the judge merely mentions both responses.
    a_found = re.search(r'\b(?:A is better|Winner:\s*A)\b', judgment, re.IGNORECASE) is not None
    b_found = re.search(r'\b(?:B is better|Winner:\s*B)\b', judgment, re.IGNORECASE) is not None
    if a_found and not b_found:
        verdict = "A"
    elif b_found and not a_found:
        verdict = "B"
    else:
        verdict = "Tie"  # explicit tie, or ambiguous judgment
    
    return {"verdict": verdict, "explanation": judgment.strip()}


# Demo: Compare two responses
# (Using the same model as judge for demonstration -- in practice, use a stronger model)
test_question = "What is the capital of France?"
response_a = "The capital of France is Paris. It is located in northern France along the Seine River and has been the country's capital since the 10th century."
response_b = "paris"

print("Judging responses...")
result = llm_judge(test_question, response_a, response_b, model, tokenizer)
print(f"Verdict: {result['verdict']}")
print(f"Explanation: {result['explanation'][:200]}")

In [ ]:
def check_position_bias(question: str, response_a: str, response_b: str,
                        judge_model, judge_tokenizer) -> dict:
    """
    Check for position bias by running the judge twice with swapped order.
    Position bias is one of the key failure modes of LLM-as-judge.
    """
    # Original order
    result_ab = llm_judge(question, response_a, response_b, judge_model, judge_tokenizer)
    
    # Swapped order
    result_ba = llm_judge(question, response_b, response_a, judge_model, judge_tokenizer)
    # Flip the verdict for comparison
    flipped = {"A": "B", "B": "A", "Tie": "Tie"}
    result_ba_flipped = flipped.get(result_ba["verdict"], "Tie")
    
    consistent = (result_ab["verdict"] == result_ba_flipped)
    
    print(f"Order A,B -> Verdict: {result_ab['verdict']}")
    print(f"Order B,A -> Verdict: {result_ba['verdict']} (flipped: {result_ba_flipped})")
    print(f"Consistent: {consistent}")
    
    if not consistent:
        print("WARNING: Position bias detected! The judge's verdict changed when we swapped the order.")
    
    return {
        "original_verdict": result_ab["verdict"],
        "swapped_verdict": result_ba["verdict"],
        "consistent": consistent
    }


# Test for position bias
print("Testing for position bias...\n")
bias_result = check_position_bias(
    "Explain photosynthesis.",
    "Photosynthesis is how plants make food from sunlight.",
    "Photosynthesis is the process by which green plants convert light energy into chemical energy. In the chloroplasts, water and CO2 are transformed into glucose and oxygen via the light-dependent and light-independent (Calvin cycle) reactions.",
    model, tokenizer
)

---
## 5. Benchmark Contamination

### What Is Contamination?

**Benchmark contamination** occurs when a model's training data includes examples from the benchmark test set. This artificially inflates performance because the model has effectively "seen the answers" rather than genuinely solving the problems.

### Why It's a Critical Problem

- A model scoring **90% on MMLU** might actually perform at **75% on truly unseen questions**
- Pre-training corpora (Common Crawl, etc.) contain benchmark datasets that were posted online
- Fine-tuning datasets often inadvertently include benchmark examples
- This makes it **impossible to compare models fairly** if contamination levels differ

### Detection Methods

1. **N-gram overlap analysis**: Check if test examples appear verbatim in training data
2. **Canary strings**: Insert unique identifiers in benchmarks; if the model can reproduce them, it saw the data
3. **Temporal splits**: Only use data created after the training cutoff (LiveCodeBench approach)
4. **Perturbation analysis**: Rephrase benchmark questions; if performance drops dramatically, contamination is likely
5. **Membership inference**: Use statistical tests to check if the model "memorized" specific examples

### Notable Incidents

- GPT-4 technical report acknowledged potential contamination on several benchmarks
- Multiple open-source models were found to have trained on HumanEval test cases
- Phi-1 was specifically criticized for potential benchmark contamination

In [ ]:
def compute_ngram_overlap(text1: str, text2: str, n: int = 8) -> float:
    """
    Compute n-gram overlap between two texts.
    High overlap suggests contamination.
    
    This is a simplified version of the contamination checks used by
    OpenAI (GPT-4 report) and other labs.
    """
    def get_ngrams(text, n):
        words = text.lower().split()
        return set(tuple(words[i:i+n]) for i in range(len(words) - n + 1))
    
    ngrams1 = get_ngrams(text1, n)
    ngrams2 = get_ngrams(text2, n)
    
    if not ngrams1 or not ngrams2:
        return 0.0
    
    overlap = ngrams1 & ngrams2
    # Jaccard similarity
    return len(overlap) / len(ngrams1 | ngrams2)


def check_contamination(benchmark_examples: list, training_corpus: list, 
                        n: int = 8, threshold: float = 0.5) -> dict:
    """
    Check if benchmark examples appear in training corpus.
    Returns contamination report.
    """
    contaminated = []
    clean = []
    
    for i, bench_example in enumerate(benchmark_examples):
        max_overlap = 0.0
        most_similar = None
        
        for train_example in training_corpus:
            overlap = compute_ngram_overlap(bench_example, train_example, n)
            if overlap > max_overlap:
                max_overlap = overlap
                most_similar = train_example
        
        if max_overlap >= threshold:
            contaminated.append({
                'index': i,
                'benchmark_text': bench_example[:100],
                'overlap': max_overlap,
                'similar_train': most_similar[:100] if most_similar else None
            })
        else:
            clean.append(i)
    
    contamination_rate = len(contaminated) / len(benchmark_examples)
    
    return {
        'total_examples': len(benchmark_examples),
        'contaminated_count': len(contaminated),
        'clean_count': len(clean),
        'contamination_rate': contamination_rate,
        'contaminated_examples': contaminated
    }


# Simulate contamination scenario
benchmark_questions = [
    "What is the capital of France? A) London B) Paris C) Berlin D) Madrid",
    "Which planet is closest to the sun? A) Venus B) Mercury C) Mars D) Earth",
    "What is the speed of light? A) 300km/s B) 300000km/s C) 3000km/s D) 30000km/s",
    "Who wrote Hamlet? A) Dickens B) Shakespeare C) Austen D) Twain",
    "What is the atomic number of carbon? A) 8 B) 6 C) 12 D) 4",
]

# Simulated training corpus -- some examples are contaminated (near-identical to benchmark)
training_corpus = [
    "What is the capital of France? A) London B) Paris C) Berlin D) Madrid. The answer is B) Paris.",  # Contaminated!
    "France is a country in Western Europe. Its capital city is Paris, known for the Eiffel Tower.",  # Clean -- different phrasing
    "Mercury is the closest planet to the sun in our solar system.",  # Clean -- no MCQ format
    "Who wrote Hamlet? A) Dickens B) Shakespeare C) Austen D) Twain. Answer: B",  # Contaminated!
    "The periodic table lists elements by atomic number. Carbon has atomic number 6.",  # Clean
]

report = check_contamination(benchmark_questions, training_corpus, n=5, threshold=0.3)

print(f"Contamination Report")
print(f"{'='*50}")
print(f"Total benchmark examples: {report['total_examples']}")
print(f"Contaminated: {report['contaminated_count']} ({report['contamination_rate']:.1%})")
print(f"Clean: {report['clean_count']}")
print(f"\nContaminated examples:")
for ex in report['contaminated_examples']:
    print(f"  [{ex['index']}] Overlap={ex['overlap']:.2f}: {ex['benchmark_text'][:60]}...")

In [ ]:
# Visualize the impact of contamination on reported vs. true accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Contamination inflates scores
contamination_levels = [0, 10, 20, 30, 50, 75, 100]
true_accuracy = 65  # Hypothetical true accuracy
contaminated_accuracy = 95  # Accuracy on contaminated (memorized) examples

reported_accuracies = [
    (1 - c/100) * true_accuracy + (c/100) * contaminated_accuracy 
    for c in contamination_levels
]

axes[0].plot(contamination_levels, reported_accuracies, 'ro-', linewidth=2, markersize=8)
axes[0].axhline(y=true_accuracy, color='blue', linestyle='--', label=f'True accuracy ({true_accuracy}%)')
axes[0].fill_between(contamination_levels, true_accuracy, reported_accuracies, alpha=0.2, color='red')
axes[0].set_xlabel('Contamination Rate (%)', fontsize=12)
axes[0].set_ylabel('Reported Accuracy (%)', fontsize=12)
axes[0].set_title('How Contamination Inflates Scores', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].set_ylim(50, 100)
axes[0].grid(True, alpha=0.3)

# Plot 2: Benchmark saturation over time
years = ['2020', '2021', '2022', '2023', '2024', '2025', '2026']
gsm8k_scores = [None, 35, 55, 80, 92, 95, 97]  # GSM8K was released Oct 2021
humaneval_scores = [15, 30, 50, 70, 85, 92, 95]
math_scores = [10, 15, 25, 45, 65, 80, 90]
frontiermath_scores = [None, None, None, None, 5, 15, 28]

gsm8k_years = [y for y, s in zip(years, gsm8k_scores) if s is not None]
gsm8k_valid = [s for s in gsm8k_scores if s is not None]
axes[1].plot(gsm8k_years, gsm8k_valid, 's-', label='GSM8K (released Oct 2021)', linewidth=2, color='red')
axes[1].plot(years, humaneval_scores, 'o-', label='HumanEval', linewidth=2, color='blue')
axes[1].plot(years, math_scores, '^-', label='MATH', linewidth=2, color='green')
valid_years = [y for y, s in zip(years, frontiermath_scores) if s is not None]
valid_scores = [s for s in frontiermath_scores if s is not None]
axes[1].plot(valid_years, valid_scores, 'D-', label='FrontierMath', linewidth=2, color='purple')
axes[1].axhline(y=90, color='gray', linestyle=':', alpha=0.5, label='Saturation zone')
axes[1].set_xlabel('Year', fontsize=12)
axes[1].set_ylabel('SOTA Accuracy (%)', fontsize=12)
axes[1].set_title('Benchmark Saturation Over Time', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Goodhart's Law for Benchmarks

> **"When a measure becomes a target, it ceases to be a good measure."** -- Charles Goodhart

### How This Applies to LLM Evaluation

When labs optimize for specific benchmarks, models learn to game them rather than genuinely improve.

### Concrete Examples

| Benchmark | Gaming Behavior | Consequence |
|-----------|----------------|-------------|
| **MMLU** | Models memorize answer patterns, learn elimination heuristics | High MMLU score does not mean deep understanding |
| **AlpacaEval** | Models learn to write **longer, more verbose responses** because GPT-4 judge prefers length | Win rate correlates with response length, not quality |
| **HumanEval** | Models memorize common Python patterns that appear in training data | High pass@1 doesn't generalize to real-world coding |
| **MT-Bench** | Models learn to produce well-structured, confident-sounding responses | Style over substance |

### The AlpacaEval Length Bias Story

This is a particularly instructive case:

1. **AlpacaEval v1** used GPT-4 to judge model outputs against a reference
2. Researchers discovered that **the GPT-4 judge systematically favors longer outputs** -- simply making responses more verbose could inflate win rate, making the metric gameable
3. **AlpacaEval 2.0 (LC)** (Dubois et al., 2024) introduced **length-controlled win rates** to correct for this
4. Length control raised AlpacaEval's correlation **with Chatbot Arena** from 0.94 to 0.98 (Dubois et al., 2024) -- note: these often-misquoted numbers are benchmark-to-Arena correlations, NOT a "0.95 length-vs-win-rate correlation"
5. After LC correction, the rankings changed significantly -- models that were "winning" by being verbose dropped in ranking

### Lessons for Interview

1. **No single benchmark is sufficient** -- always evaluate on a diverse suite
2. **Held-out evaluations** (like Chatbot Arena with live users) are more robust than static benchmarks
3. **Be skeptical of benchmark improvements** -- ask "did the model genuinely improve, or did it learn to game this metric?"
4. **Dynamic benchmarks** (LiveCodeBench, FrontierMath) that update over time resist gaming
5. **Process metrics** (how the model solves problems) may be more informative than outcome metrics (whether it got the right answer)

In [ ]:
# Demonstrate the length bias problem
# Simulating how longer responses tend to be preferred by LLM judges

def simulate_length_bias_experiment():
    """
    Simulate the AlpacaEval length bias finding.
    Shows that response length correlates with win rate.
    """
    np.random.seed(42)
    
    # Simulate 100 model responses with varying lengths and true quality scores
    n_responses = 100
    
    # True quality (0-1) -- what we actually care about
    true_quality = np.random.beta(2, 2, n_responses)  # Centered around 0.5
    
    # Response length (in tokens)
    response_lengths = np.random.lognormal(5, 0.8, n_responses).astype(int)
    response_lengths = np.clip(response_lengths, 50, 2000)
    
    # Simulated judge preference -- biased toward length
    # Judge score = 0.3 * true_quality + 0.7 * normalized_length + noise
    normalized_length = (response_lengths - response_lengths.min()) / (response_lengths.max() - response_lengths.min())
    judge_score = 0.3 * true_quality + 0.7 * normalized_length + np.random.normal(0, 0.05, n_responses)
    judge_score = np.clip(judge_score, 0, 1)
    
    # Length-controlled judge score -- removes length bias
    lc_judge_score = judge_score - 0.5 * normalized_length  # Simple debiasing
    lc_judge_score = (lc_judge_score - lc_judge_score.min()) / (lc_judge_score.max() - lc_judge_score.min())
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Length vs raw judge score (high correlation = bias)
    axes[0].scatter(response_lengths, judge_score, alpha=0.5, c='red')
    z = np.polyfit(response_lengths, judge_score, 1)
    p = np.poly1d(z)
    x_line = np.linspace(response_lengths.min(), response_lengths.max(), 100)
    axes[0].plot(x_line, p(x_line), 'k--', linewidth=2)
    corr = np.corrcoef(response_lengths, judge_score)[0, 1]
    axes[0].set_title(f'Raw Judge Score vs Length\n(r={corr:.2f} -- STRONG BIAS)', fontsize=13)
    axes[0].set_xlabel('Response Length (tokens)')
    axes[0].set_ylabel('Judge Win Rate')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Length vs LC judge score (reduced correlation)
    axes[1].scatter(response_lengths, lc_judge_score, alpha=0.5, c='blue')
    z2 = np.polyfit(response_lengths, lc_judge_score, 1)
    p2 = np.poly1d(z2)
    axes[1].plot(x_line, p2(x_line), 'k--', linewidth=2)
    corr2 = np.corrcoef(response_lengths, lc_judge_score)[0, 1]
    axes[1].set_title(f'LC Judge Score vs Length\n(r={corr2:.2f} -- bias reduced)', fontsize=13)
    axes[1].set_xlabel('Response Length (tokens)')
    axes[1].set_ylabel('LC Win Rate')
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: True quality vs each judge score
    axes[2].scatter(true_quality, judge_score, alpha=0.5, c='red', label=f'Raw (r={np.corrcoef(true_quality, judge_score)[0,1]:.2f})')
    axes[2].scatter(true_quality, lc_judge_score, alpha=0.5, c='blue', label=f'LC (r={np.corrcoef(true_quality, lc_judge_score)[0,1]:.2f})')
    axes[2].set_title('Which Score Better Reflects True Quality?', fontsize=13)
    axes[2].set_xlabel('True Quality')
    axes[2].set_ylabel('Judge Score')
    axes[2].legend(fontsize=11)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nKey Insight: In this SIMULATION, raw judge scores are dominated by length.")
    print("Length-controlled scores better reflect actual response quality.")
    print("This is why AlpacaEval 2.0 introduced LC win rates: GPT-4 judges favor longer")
    print("outputs, making the raw metric gameable via verbosity. Length control raised")
    print("AlpacaEval's correlation with Chatbot Arena from 0.94 to 0.98 (Dubois et al., 2024).")


simulate_length_bias_experiment()

---
## 7. LLM-as-Judge: Deep Dive

### The Core Idea

Use a **strong LLM** (e.g., GPT-4, Claude 4.6) to evaluate the outputs of **weaker models**. This is:
- Much cheaper than human evaluation
- More scalable (can evaluate thousands of examples)
- Reasonably well-correlated with human preferences (~80% agreement)

### MT-Bench Approach (Zheng et al., 2023)

1. **80 multi-turn questions** across 8 categories (writing, roleplay, reasoning, math, coding, extraction, STEM, humanities)
2. **GPT-4 as judge**: scores each response on a 1-10 scale
3. **Two evaluation modes**:
   - **Single-answer grading**: judge scores one response independently
   - **Pairwise comparison**: judge compares two responses and picks the better one
4. **Finding**: GPT-4 judge agrees with human preferences >80% of the time

### Known Biases

| Bias | Description | Mitigation |
|------|-------------|------------|
| **Position bias** | Judge prefers the response shown first (or second, depending on model) | Run twice with swapped order; take majority vote |
| **Verbosity bias** | Judge prefers longer, more detailed responses regardless of accuracy | Length-controlled metrics (AlpacaEval 2.0 LC) |
| **Self-preference bias** | GPT-4 judge prefers GPT-4 outputs over other models' outputs | Use diverse judge panel; weight by consistency |
| **Style bias** | Judge prefers well-formatted, confident-sounding responses | Focus evaluation criteria on factual accuracy |
| **Sycophancy** | Judge agrees with whatever the response says rather than checking correctness | Provide reference answers; use fact-checking |

### When LLM-as-Judge Works Well

- **Open-ended questions** where there's no single right answer
- **Style and helpfulness** evaluation
- **Comparing models of similar capability level**

### When LLM-as-Judge Fails

- **Math/logic problems** where the judge can't verify correctness
- **Evaluating models stronger than the judge** (ceiling effect)
- **Domain-specific tasks** the judge doesn't understand
- **Subtle factual errors** that sound plausible

In [ ]:
def mt_bench_style_evaluation(responses: list, categories: list, 
                               judge_model, judge_tokenizer) -> pd.DataFrame:
    """
    Implement MT-Bench style single-answer grading.
    Judge scores each response on a 1-10 scale.
    """
    results = []
    
    for resp, category in zip(responses, categories):
        prompt = f"""Rate the following AI assistant response on a scale of 1 to 10.
Consider: accuracy, helpfulness, depth, and clarity.

Category: {category}
Question: {resp['question']}
Response: {resp['response']}

Score (1-10): Just output the number."""
        
        inputs = judge_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(judge_model.device)
        
        with torch.no_grad():
            outputs = judge_model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.1,
                do_sample=True,
                pad_token_id=judge_tokenizer.eos_token_id
            )
        
        score_text = judge_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        # Extract score
        score_match = re.search(r'(\d+)', score_text)
        score = int(score_match.group(1)) if score_match else 5
        score = min(max(score, 1), 10)  # Clamp to 1-10
        
        results.append({
            'category': category,
            'question': resp['question'][:50] + '...',
            'score': score,
            'response_length': len(resp['response'].split())
        })
    
    return pd.DataFrame(results)


# Example responses to judge
test_responses = [
    {
        'question': 'What is machine learning?',
        'response': 'Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It uses algorithms to find patterns in data and make predictions or decisions.'
    },
    {
        'question': 'Write a haiku about coding.',
        'response': 'Fingers on the keys\nLogic flows like mountain streams\nBugs hide in the mist'
    },
    {
        'question': 'Explain the Pythagorean theorem.',
        'response': 'a^2 + b^2 = c^2. In a right triangle, the square of the hypotenuse equals the sum of squares of the other two sides. This was proven by Pythagoras around 500 BCE and has countless applications in geometry, physics, and engineering.'
    },
]
test_categories = ['STEM', 'Writing', 'Math']

print("Running MT-Bench style evaluation...")
eval_df = mt_bench_style_evaluation(test_responses, test_categories, model, tokenizer)
print("\nResults:")
print(eval_df.to_string(index=False))
print(f"\nAverage score: {eval_df['score'].mean():.1f}/10")

---
## 8. "Why Does This Work?" -- Current Debates

### Are Benchmarks Meaningful?

**Arguments that benchmarks are useful:**
- They provide a common language for comparing models
- Chatbot Arena correlates well with real-world usefulness
- Diverse benchmark suites (like the Open LLM Leaderboard) give a multifaceted view
- Benchmarks drive progress by setting clear targets

**Arguments that benchmarks are misleading:**
- Contamination makes many benchmarks unreliable
- Goodhart's Law means optimization for benchmarks diverges from true capability
- Static benchmarks can't capture the full range of model capabilities
- MCQ format (MMLU) doesn't reflect real-world usage
- Models can score well by exploiting surface patterns, not deep understanding

### The Path Forward

1. **Dynamic benchmarks** that update continuously (LiveCodeBench, Chatbot Arena)
2. **Private benchmarks** that models can't be trained on (FrontierMath, SEAL Leaderboard)
3. **Process-based evaluation** -- evaluate HOW the model solves problems, not just the final answer
4. **Real-world task evaluation** -- measure actual downstream task performance
5. **Human evaluation at scale** -- Chatbot Arena approach, but extended to more domains
6. **Adversarial evaluation** -- continuously try to break models, measure robustness

### Interview Tip

When asked about evaluation in interviews, demonstrate:
- Awareness that **no single benchmark is sufficient**
- Understanding of **contamination and its implications**
- Knowledge of the **strengths and weaknesses of LLM-as-judge**
- Familiarity with **current best practices** (diverse eval suite, Chatbot Arena, dynamic benchmarks)
- Opinion on **what makes a good benchmark** (unsaturated, contamination-resistant, well-calibrated, diverse)

---
## Interview Gauntlet: Evaluation & Benchmarks

*These are representative, illustrative questions of the kind asked in senior/principal ML interviews at frontier labs. The rubric reflects what separates "hire" from "strong hire."*

---

### Question 1: "How would you evaluate a new LLM? Design the evaluation pipeline."

**What the interviewer is testing:** Do you understand that benchmarks are necessary but insufficient? Can you design a production-grade evaluation system?

**Good answer** (meets bar): Describes standard benchmarks (MMLU, HumanEval, GSM8K), mentions human evaluation, discusses the importance of multiple evaluation dimensions.

**Great answer** (strong hire): Designs the full pipeline with layers:

1. **Automated benchmark suite** -- standard benchmarks for regression detection, run on every training checkpoint
2. **Internal eval set** -- 500-2000 carefully curated prompts that are NEVER published, spanning categories (reasoning, coding, creative writing, safety, multilingual). These are the prompts that actually drive decisions.
3. **LLM-as-judge** -- automated quality scoring with bias controls (position randomization, multiple judges, calibration against human labels)
4. **Human evaluation** -- expert raters score 1-5 on specific dimensions, tracked over model versions. Inter-rater reliability measured via Cohen's kappa.
5. **Red teaming** -- dedicated adversarial testing to probe for failures before launch
6. **A/B testing** -- 1-5% of traffic sees new model, measure user preference, retention, task completion, safety metrics
7. **Staged rollout** -- canary deployment with automated regression alerts

The pipeline architecture: `checkpoint --> automated benchmarks (1 hour) --> internal eval (4 hours) --> human spot-check (1-2 days) --> red team (1-2 weeks) --> staged rollout (1-2 weeks)`

**Follow-up: "Your model scores 95 on MMLU but users say it's worse than the previous version. Explain."**

This tests whether you understand the gap between benchmarks and user satisfaction:
- MMLU measures knowledge retrieval; users care about instruction following, tone, helpfulness, and format
- The model may have improved on factuality but regressed on conversational quality (sycophancy, verbosity, hedging)
- Benchmark contamination: the model may have memorized MMLU questions during training
- Distribution shift: your users' actual queries may look nothing like MMLU
- The 95 vs 93 difference might be within noise; the user experience regression is in an unmeasured dimension
- **What to do:** Run your internal eval set (which tests real user-like queries), check Chatbot Arena if available, run A/B test with explicit user preference signals

**Follow-up: "Design an evaluation that's resistant to gaming and contamination."**

Strong answer includes:
- **Private held-out sets** that are never published or included in any training data
- **Regularly refreshed questions** -- new questions generated monthly, old ones retired
- **Canary strings** -- embed unique identifiers in eval questions to detect if they leak into training data
- **Format variation** -- same underlying task presented in many different surface forms
- **N-gram overlap detection** -- check training data against eval questions (8-gram overlap threshold)
- **Live evaluation** -- Chatbot Arena style, where questions come from real users in real-time

### Question 2: "LLM-as-judge vs human evaluation -- design a study comparing them."

**What the interviewer is testing:** Can you design rigorous experimental methodology? Do you know the known biases?

**Good answer:** Describes both approaches, mentions they sometimes disagree, suggests comparing agreement rates.

**Great answer** designs a study:

**Study Design:**
1. **Sample:** 500+ diverse prompts across 8+ categories (reasoning, creative, safety, coding, multilingual, etc.)
2. **Responses:** Collect outputs from 3-5 different models at different capability levels
3. **Human evaluation:** 3+ raters per example, measure inter-rater reliability (Cohen's kappa > 0.6 target), clear rubric with anchor examples, compensated fairly to avoid rushed ratings
4. **LLM-as-judge evaluation:** Run 3+ judge models, randomize presentation order (position bias control), run each comparison twice with swapped positions
5. **Metrics:** Agreement rate, Cohen's kappa between human and LLM judge, category-level breakdown, failure case analysis

**Known biases to discuss:**
- **Position bias:** LLM judges tend to prefer the first response (~60% preference). Mitigation: swap positions and average.
- **Verbosity bias:** Longer responses rated higher regardless of quality. Mitigation: length-controlled comparisons.
- **Self-preference bias:** GPT-4 as judge prefers GPT-4 outputs over Claude, and vice versa. Mitigation: use multiple diverse judges.
- **Calibration drift:** LLM judges may not maintain consistent standards across sessions. Mitigation: include calibration examples.
- **Sycophancy:** LLM judges may agree with confidently stated but wrong answers. Mitigation: include ground-truth factuality checks.

**Statistical methodology:** Bootstrap confidence intervals on agreement rates. Report per-category agreement since overall agreement can mask category-level disagreements.

---

### Question 3: "What's wrong with current benchmarks?"

**What the interviewer is testing:** Critical thinking about evaluation methodology. Research maturity.

**Good answer:** Contamination, saturation (models scoring near ceiling), limited coverage.

**Great answer** discusses structural problems:

1. **Goodhart's Law** -- "When a measure becomes a target, it ceases to be a good measure." Labs optimize for benchmarks, which stops benchmarks from measuring what they were designed to measure.
2. **Format-specific overfitting** -- Models learn to recognize benchmark question formats rather than actually solving the underlying task. Performance on slightly reformatted questions drops significantly.
3. **Contamination is endemic** -- with internet-scale training data, it is nearly impossible to guarantee zero contamination. Even "new" benchmarks get contaminated within months of release.
4. **Saturation** -- many benchmarks are near-ceiling (MMLU, HellaSwag, ARC), making them useless for differentiating frontier models.
5. **Lack of real-world validity** -- the correlation between benchmark scores and actual user satisfaction is weak. A model can score 95 on MMLU and be frustrating to use.
6. **Static vs dynamic** -- benchmarks capture a frozen snapshot of what we thought was important. The actual frontier of capability moves faster than benchmark design.
7. **Gaming without cheating** -- even without contamination, training on benchmark-like data distributions (same format, same reasoning patterns) inflates scores without improving general capability.

**The meta-point:** Evaluation is arguably the hardest unsolved problem in LLM development. If you can't measure it, you can't improve it -- and we're bad at measuring what matters.

### Production Notes: How a Production Eval Pipeline Typically Works

*An illustrative picture of common industry practice -- not internal details of any specific lab.*

- **Internal eval sets:** 500-2000 carefully curated prompts across categories. These are the REAL evaluation -- not MMLU. They are never published. New prompts are added continuously.
- **Human evaluators:** Trained raters (often contractors via Scale AI, Surge AI, or internal teams). Rate on 1-5 scales across dimensions (helpfulness, accuracy, safety, format). Inter-rater reliability tracked via Cohen's kappa.
- **Eval cadence:** Automated benchmarks run on EVERY checkpoint during training. Internal eval runs daily or per-major-checkpoint. Human eval runs for release candidates only.
- **A/B testing:** 1-5% of production traffic sees the new model. Key metrics: user preference (thumbs up/down), task completion rate, conversation length, retention (do users come back?), safety incident rate.
- **Red teaming:** Dedicated red teamers (internal and external) probe release candidates before launch. Their job is to find the worst possible outputs. Findings go into the system card / model card.
- **Automated regression alerts:** If any benchmark drops more than X% from previous checkpoint, training is flagged for review. This catches catastrophic forgetting early.
- **The honest truth:** Even with all of this, models sometimes launch and users report problems that the eval pipeline missed. The eval pipeline is continuously improved based on post-launch feedback.

---
## 9. Flashcard Summary

Use these for spaced repetition review. Cover the answer and test yourself.

| # | Question | Answer |
|---|----------|--------|
| 1 | What is MMLU and what are its limitations? | Massive Multitask Language Understanding: 57-subject MCQ benchmark. Limitations: largely saturated (~90%+), contamination concerns, noisy labels (~6% error rate), MCQ format doesn't reflect real usage. |
| 2 | Why is Chatbot Arena considered the gold standard? | Uses blind A/B comparisons with real users, producing Bradley-Terry-based Arena Scores (colloquially still called "Elo" -- LMArena replaced online Elo with a Bradley-Terry MLE fit in Dec 2023). Resistant to contamination (live data), reflects real-world preferences, and constantly updated. |
| 3 | What is benchmark contamination? | When training data includes benchmark test examples, artificially inflating scores. A model might score 90% on MMLU but only 75% on truly unseen questions. |
| 4 | State Goodhart's Law and give an LLM example. | "When a measure becomes a target, it ceases to be a good measure." Example: optimizing for AlpacaEval led models to write longer responses because the GPT-4 judge preferred verbosity, not quality. |
| 5 | What is the difference between GSM8K and FrontierMath? | GSM8K: grade-school math, saturated (~97%). FrontierMath: expert-level research math, still unsaturated (~52% on Tiers 1-3 as of mid-2026), designed to stay hard for years. |
| 6 | What is SWE-bench? Why is it harder than HumanEval? | SWE-bench tests fixing real GitHub issues requiring understanding of entire codebases. HumanEval is simple function generation from docstrings (saturated at 95%+). |
| 7 | Name 3 biases in LLM-as-judge. | Position bias (prefers first/second response), verbosity bias (prefers longer responses), self-preference bias (GPT-4 prefers GPT-4 outputs). |
| 8 | What is AlpacaEval LC and why was it created? | Length-Controlled win rate version of AlpacaEval. Created because the GPT-4 judge favors longer outputs, making the raw metric gameable via verbosity. Length control raised AlpacaEval's correlation with Chatbot Arena from 0.94 to 0.98 (Dubois et al., 2024). |
| 9 | How do you detect benchmark contamination? | N-gram overlap analysis, canary strings, temporal splits (only test on data after training cutoff), perturbation analysis (rephrase questions), membership inference. |
| 10 | What is GPQA and why is it significant? | Graduate-level PhD science questions. Significant because human domain experts only score ~65%, making it one of the few benchmarks where models approach or exceed expert-level. |
| 11 | What is IFEval? | Instruction-Following Eval: tests whether models follow precise formatting constraints (e.g., "respond in exactly 3 sentences"). Tests surface compliance, not understanding. |
| 12 | What is SimpleQA and what does it measure? | OpenAI's benchmark for factual accuracy on simple questions. Measures hallucination rate. Many models score only 40-50%, showing even simple facts cause hallucinations. |
| 13 | What makes a good LLM benchmark in 2026? | Unsaturated, contamination-resistant (temporal cutoffs or private), diverse question types, well-calibrated difficulty, measures genuine capability not pattern matching. |
| 14 | What is MT-Bench and how does it work? | 80 multi-turn questions across 8 categories. GPT-4 judges responses on 1-10 scale. Agrees with human preferences >80% of the time. Becoming saturated (~9.5/10 for frontier models). |
| 15 | What is Humanity's Last Exam? | A grand challenge benchmark of expert-level questions across all academic disciplines, designed to be extremely hard for AI. Frontier models score ~53% without tools / ~64% with tools (June 2026). |

---
## 10. Paper Guide

### Primary Paper: "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena"

**Citation**: Zheng et al., 2023. "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena." NeurIPS 2023.

**Link**: [https://arxiv.org/abs/2306.05685](https://arxiv.org/abs/2306.05685)

### Why Read This Paper?

This paper introduced two of the most influential evaluation methods in the LLM era and is a must-know for frontier lab interviews.

### Key Contributions

1. **MT-Bench**: A curated set of 80 multi-turn questions designed to test specific capabilities
2. **Chatbot Arena**: A crowdsourced platform where users compare model responses in blind A/B tests
3. **Systematic study of LLM-as-judge**: When it works, when it fails, and how to mitigate biases

### Reading Guide (3-Pass Method)

**Pass 1 (30 min)**: Read abstract, introduction, and conclusion. Understand the problem: how do we evaluate open-ended LLM responses at scale?

**Pass 2 (1 hour)**: Focus on:
- Section 3: MT-Bench design and question categories
- Section 4: Chatbot Arena methodology
- Section 5: LLM judge agreement analysis -- especially Tables 3-5 showing agreement rates
- Section 6: Position bias and consistency analysis

**Pass 3 (1 hour)**: Deep dive into:
- The failure cases of LLM-as-judge (when does GPT-4 disagree with humans?)
- The Arena rating methodology (the paper used online Elo; in Dec 2023 LMArena switched to a Bradley-Terry MLE fit, though scores are colloquially still called "Elo")
- How they handle ties and inconsistencies

### Key Findings to Remember

1. Strong LLMs (GPT-4) match human inter-annotator agreement rates (~80%)
2. Position bias is real but can be mitigated by running evaluations with swapped order
3. Pairwise comparison is more reliable than single-answer grading
4. Math and reasoning questions are hardest for LLM judges
5. Chatbot Arena's Bradley-Terry-based Arena Scores (colloquially still called "Elo") have become the de facto standard for model comparison

### Related Papers

- **AlpacaEval 2.0** (Dubois et al., 2024): Length-controlled LLM evaluation
- **Arena-Hard** (Li et al., 2024): Automated benchmark derived from Chatbot Arena
- **LiveCodeBench** (Jain et al., 2024): Contamination-free code benchmark with temporal cutoffs
- **FrontierMath** (Glazer et al., 2024): Expert-level math benchmark designed to be unsaturated
- **MMLU-Pro** (Wang et al., 2024): Harder version of MMLU with 10 answer choices

### Additional Recent Papers (2024-2025)

- **SimpleQA** (OpenAI, 2024): Tests factual accuracy with simple questions. Reveals that even frontier models hallucinate 10-40% on easy factual queries. [https://openai.com/index/introducing-simpleqa/](https://openai.com/index/introducing-simpleqa/)
- **LiveCodeBench** (Jain et al., 2024): Continuously updated coding benchmark using problems posted after training cutoffs. Resistant to contamination by design.
- **FrontierMath** (Glazer et al., 2024): Research-level mathematics benchmark designed to remain unsaturated for years.
- **GPQA** (Rein et al., 2023): PhD-level science questions where domain experts score ~65%. Tests genuine graduate-level reasoning.


---

**End of Notebook 19**

You should now be able to:
- Name and describe the major LLM benchmarks across all categories
- Build a simple eval harness with answer extraction and exact-match scoring
- Implement and critique LLM-as-judge evaluation
- Explain benchmark contamination and its detection methods
- Apply Goodhart's Law to benchmark optimization
- Discuss the strengths and limitations of current evaluation approaches

**Next**: Notebook 20 covers Safety, Red-Teaming & Alignment Failure Modes.